In [ ]:
# -----------------------------
# STEP 0: Install
# -----------------------------
!pip install yt-dlp opencv-python matplotlib torch torchvision
!pip uninstall -y clip
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
import sys # Import sys module to manage imported modules

# Load CLIP
# Ensure no stale 'clip' module is in sys.modules before importing
if 'clip' in sys.modules:
    del sys.modules['clip']
import clip

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

# -----------------------------
# STEP 1: Download video
# -----------------------------
video_url = "https://www.youtube.com/watch?v=m9coOXt5nuw"

!yt-dlp -f mp4 -o video.mp4 {video_url}
video_path = "video.mp4"

In [ ]:
# -----------------------------
# STEP 2: Extract frames
# -----------------------------
cap = cv2.VideoCapture(video_path)

frames = []
success, frame = cap.read()

while success:
    frames.append(frame)
    success, frame = cap.read()

cap.release()
print("Total frames:", len(frames))

In [ ]:
# -----------------------------
# STEP 3: Compute CLIP embeddings
# -----------------------------
def get_embedding(frame):
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    image = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        embedding = model.encode_image(image)

    return embedding.cpu().numpy().flatten()

embeddings = []

for i, frame in enumerate(frames):
    if i % 5 == 0:  # sample every 5 frames (speed)
        emb = get_embedding(frame)
        embeddings.append(emb)

embeddings = np.array(embeddings)

In [ ]:
# -----------------------------
# STEP 4: Compute similarity
# -----------------------------
from sklearn.metrics.pairwise import cosine_similarity

similarities = []

for i in range(1, len(embeddings)):
    sim = cosine_similarity([embeddings[i-1]], [embeddings[i]])[0][0]
    similarities.append(sim)

# -----------------------------
# STEP 5: Detect scene changes
# -----------------------------
threshold = np.mean(similarities) - 0.2

scene_changes = [i for i, s in enumerate(similarities) if s < threshold]

print("Scene changes:", scene_changes)

# -----------------------------
# STEP 6: Extract keyframes
# -----------------------------
os.makedirs("clip_keyframes", exist_ok=True)

cap = cv2.VideoCapture(video_path)

for idx in scene_changes:
    frame_id = idx * 5  # because we sampled every 5 frames
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
    success, frame = cap.read()

    if success:
        cv2.imwrite(f"clip_keyframes/frame_{frame_id}.jpg", frame)

cap.release()



In [ ]:
# -----------------------------
# STEP 7: Show keyframes
# -----------------------------
import glob

imgs = sorted(glob.glob("clip_keyframes/*.jpg"))

plt.figure(figsize=(15,5))

for i, img_path in enumerate(imgs[:5]):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(1, 5, i+1)
    plt.imshow(img)
    plt.axis("off")

plt.show()